# Acoustic Levitator — Python Simulation

Simulation of a **72-transducer, 40 kHz ultrasonic acoustic levitator**
(TinyLev-style: 36 transducers in a top shell + 36 in a bottom shell),
including the acoustic pressure field, the **Gor'kov radiation potential**,
and **phase-control recipes** to move, rotate and shape the trapped particle.

This notebook accompanies `acoustic_levitator.py` and the `README.md`.
Run the cells top to bottom.  Dependencies: `numpy`, `scipy`, `matplotlib`.

> **What you can do here**
> 1. Visualise the standing-wave trap (the classic levitator).
> 2. Translate the trap up/down with the top–bottom phase offset.
> 3. Spin a particle with an **acoustic vortex** (orbital angular momentum).
> 4. Explore the two group configurations you are building:
>    - **Flower petals** — six sectors of six transducers per shell.
>    - **Concentric rings** — three rings (6 / 12 / 18) per shell.


In [ ]:
%matplotlib inline
import os, sys
import numpy as np
import matplotlib.pyplot as plt

# Make sure the module (same folder as this notebook) is importable.
sys.path.insert(0, os.getcwd())

from acoustic_levitator import (
    LevitatorArray, AcousticField, PhaseControl,
    POLYSTYRENE, AIR,
)

mm = 1e-3
print("Imports OK")


## 1. Build the array and inspect the geometry

The default geometry uses a spherical-cap curvature radius of 90 mm and a
10 mm piston diameter (typical of the TM-2425H13T/R transducers).  Each
shell holds **3 concentric rings** of 6 / 12 / 18 transducers = 36, so the
two shells give **72** sources facing the origin.


In [ ]:
lev = LevitatorArray(curvature_radius=0.090, piston_radius=5*mm)
field = AcousticField(lev)
ctrl = PhaseControl(lev)

lam = AIR["c"] / lev.freq
print(f"Frequency   : {lev.freq/1e3:.1f} kHz")
print(f"Wavelength  : {lam*1e3:.2f} mm  (lambda/2 = {lam*1e3/2:.2f} mm)")
print(f"Transducers : {len(lev.transducers)}  (top {len(lev.top)}, bottom {len(lev.bottom)})")
print(f"k           : {lev.k:.1f} rad/m")
print(f"Top shell rings: {lev._RING_COUNTS} elements per ring")

# 3-D scatter of the array geometry
fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
for tr in lev.transducers:
    ax.scatter(*tr.position*1e3, c="tab:red" if tr.position[2] > 0 else "tab:blue", s=12)
ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]"); ax.set_zlabel("z [mm]")
ax.set_title("72-transducer levitator geometry (top red / bottom blue)")
plt.tight_layout(); plt.show()


## 2. Standing wave — the classic trap

Drive the two shells in **anti-phase** so they form a standing wave.  The
Gor'kov potential $U$ is minimised at pressure nodes, where solid particles
(e.g. polystyrene) are trapped.  For a 40 kHz standing wave the nodes are
spaced by $\lambda/2 \approx 4.3$ mm.

The radiation force on a small spherical particle is $\mathbf{F} = -\nabla U$.


In [ ]:
lev.assign_groups("whole")
lev.set_phase(ctrl.standing_wave(0.0))

# X-Z axial slice through the centre (y = 0)
xs = np.linspace(-25*mm, 25*mm, 300)
zs = np.linspace(-25*mm, 25*mm, 300)
X, Z = np.meshgrid(xs, zs, indexing="ij")
Y = np.zeros_like(X)

p = field.pressure(X, Y, Z)
U, F = field.gorkov(X, Y, Z, particle=POLYSTYRENE, particle_radius=1*mm)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].pcolormesh(X*1e3, Z*1e3, np.abs(p)/np.abs(p).max(), cmap="inferno", shading="auto")
axes[0].set_title("|p| normalised")
axes[0].set_xlabel("x [mm]"); axes[0].set_ylabel("z [mm]")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(X*1e3, Z*1e3, U/np.abs(U).max(), cmap="RdBu_r", shading="auto")
axes[1].set_title("Gor'kov potential U  (blue = trap)")
axes[1].set_xlabel("x [mm]"); axes[1].set_ylabel("z [mm]")
plt.colorbar(im1, ax=axes[1])

s = 14
axes[2].quiver(X[::s, ::s]*1e3, Z[::s, ::s]*1e3,
               F[0, ::s, ::s], F[2, ::s, ::s], color="k")
axes[2].set_title("F = -grad U  (arrows point into traps)")
axes[2].set_xlabel("x [mm]"); axes[2].set_ylabel("z [mm]")

fig.suptitle("Standing-wave trap (polystyrene, 1 mm bead)")
plt.tight_layout(); plt.show()


## 3. Moving the trap with a phase offset

A relative phase $\delta$ between the top and bottom arrays shifts the
standing-wave nodes along the axis — this is exactly how a TinyLev moves a
particle **up and down** without moving the hardware.  A full $2\pi$ shift
moves the pattern by one wavelength.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, delta in zip(axes, [0.0, np.pi/4, np.pi/2]):
    lev.set_phase(ctrl.standing_wave(delta))
    U, _ = field.gorkov(X, Y, Z, particle=POLYSTYRENE, particle_radius=1*mm)
    im = ax.pcolormesh(X*1e3, Z*1e3, U/np.abs(U).max(), cmap="RdBu_r", shading="auto")
    ax.set_title(f"delta = {delta:.2f} rad")
    ax.set_xlabel("x [mm]"); ax.set_ylabel("z [mm]")
    plt.colorbar(im, ax=ax)
fig.suptitle("Axial translation via top/bottom phase offset")
plt.tight_layout(); plt.show()


## 4. Rotation — acoustic vortex (orbital angular momentum)

Applying an azimuthal phase $\phi_j = m\,\mathrm{atan2}(y_j, x_j)$ creates
a **phase singularity** on the axis (the *vortex core*).  The resulting field
carries orbital angular momentum (topological charge $m$) and forms a
**ring-shaped trap** that spins levitated particles.  $m = \pm 1$ sets the
spin direction.


In [ ]:
lev.set_phase(ctrl.vortex(m=1, shell="both"))

# Transverse slice in the levitation plane (z = 0)
xs = np.linspace(-20*mm, 20*mm, 250)
ys = np.linspace(-20*mm, 20*mm, 250)
Xx, Yy = np.meshgrid(xs, ys, indexing="ij")
Zz = np.zeros_like(Xx)

p_xy = field.pressure(Xx, Yy, Zz)
U_xy, F_xy = field.gorkov(Xx, Yy, Zz, particle=POLYSTYRENE, particle_radius=1*mm)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = axes[0].pcolormesh(Xx*1e3, Yy*1e3, np.abs(p_xy)/np.abs(p_xy).max(), cmap="inferno", shading="auto")
axes[0].set_title("|p| in levitation plane (vortex m=1)")
axes[0].set_xlabel("x [mm]"); axes[0].set_ylabel("y [mm]")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(Xx*1e3, Yy*1e3, U_xy/np.abs(U_xy).max(), cmap="RdBu_r", shading="auto")
s = 8
axes[1].quiver(Xx[::s, ::s]*1e3, Yy[::s, ::s]*1e3,
               F_xy[0, ::s, ::s], F_xy[1, ::s, ::s], color="k")
axes[1].set_title("Gor'kov U + force (ring trap)")
axes[1].set_xlabel("x [mm]"); axes[1].set_ylabel("y [mm]")
plt.colorbar(im1, ax=axes[1])

fig.suptitle("Vortex phase -> ring trap with orbital angular momentum")
plt.tight_layout(); plt.show()


## 5. Flower-petal grouping (six sectors of six)

Your first group configuration.  Each shell is split into **6 angular sectors
of 60°**, with 6 transducers each.  Here we colour each petal with a distinct
phase to show the pattern, then compute the resulting field.

*What it is good for*: lateral beam steering / tilt of the trap by applying a
phase ramp across petals, and shape control by treating each sector as one
independent holographic channel.


In [ ]:
lev.assign_groups("petals")
petal_phases = {g: (np.pi/3)*g for g in lev.group_ids("top")}
lev.set_phase(petal_phases, shell="top")
lev.set_phase(ctrl.standing_wave(0.0), shell="bottom")

top_pos = np.array([t.position for t in lev.top])
top_ph  = np.array([t.phase for t in lev.top])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sc = axes[0].scatter(top_pos[:,0]*1e3, top_pos[:,1]*1e3, c=top_ph, cmap="twilight", s=90)
axes[0].set_aspect("equal")
axes[0].set_title("Top shell — flower petals (6 sectors x 6)")
axes[0].set_xlabel("x [mm]"); axes[0].set_ylabel("y [mm]")
plt.colorbar(sc, ax=axes[0], label="phase [rad]")

U, _ = field.gorkov(X, Y, Z, particle=POLYSTYRENE, particle_radius=1*mm)
im1 = axes[1].pcolormesh(X*1e3, Z*1e3, U/np.abs(U).max(), cmap="RdBu_r", shading="auto")
axes[1].set_title("Gor'kov potential (petal phase pattern)")
axes[1].set_xlabel("x [mm]"); axes[1].set_ylabel("z [mm]")
plt.colorbar(im1, ax=axes[1])

fig.suptitle("Flower-petal grouping")
plt.tight_layout(); plt.show()

print("Groups per shell:", lev.group_ids("top"))
print("Transducers per group:", {g: int(np.sum(np.array([t.group for t in lev.top]) == g)) for g in lev.group_ids("top")})


## 6. Concentric-ring grouping (3 rings per shell)

Your second configuration.  Each shell keeps its **3 concentric rings**
(inner 6, middle 12, outer 18).  Driving each ring with an independent phase
lets you shape the axial focal length and excite different radial modes.

*What it is good for*: tuning the vertical position and **focus quality**,
and controlling radial pressure gradients (which set trapping stiffness).


In [ ]:
lev.assign_groups("rings")
ring_phases = {g: (np.pi/2)*g for g in lev.group_ids("top")}
lev.set_phase(ring_phases, shell="top")
lev.set_phase(ctrl.standing_wave(0.0), shell="bottom")

top_pos = np.array([t.position for t in lev.top])
top_ph  = np.array([t.phase for t in lev.top])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sc = axes[0].scatter(top_pos[:,0]*1e3, top_pos[:,1]*1e3, c=top_ph, cmap="twilight", s=90)
axes[0].set_aspect("equal")
axes[0].set_title("Top shell — concentric rings (3 rings)")
axes[0].set_xlabel("x [mm]"); axes[0].set_ylabel("y [mm]")
plt.colorbar(sc, ax=axes[0], label="phase [rad]")

U, _ = field.gorkov(X, Y, Z, particle=POLYSTYRENE, particle_radius=1*mm)
im1 = axes[1].pcolormesh(X*1e3, Z*1e3, U/np.abs(U).max(), cmap="RdBu_r", shading="auto")
axes[1].set_title("Gor'kov potential (ring phase pattern)")
axes[1].set_xlabel("x [mm]"); axes[1].set_ylabel("z [mm]")
plt.colorbar(im1, ax=axes[1])

fig.suptitle("Concentric-ring grouping")
plt.tight_layout(); plt.show()

print("Rings:", lev.group_ids("top"))
print("Transducers per ring:", {g: int(np.sum(np.array([t.group for t in lev.top]) == g)) for g in lev.group_ids("top")})


## 7. Physics summary (Gor'kov model)

Each transducer is a **baffled circular piston**; its complex pressure at a
field point $\mathbf{r}$ is

$$p_j(\mathbf{r}) = A_j\, D(\theta_j)\, \frac{e^{i(k|\mathbf{r}-\mathbf{r}_j| + \phi_j)}}{|\mathbf{r}-\mathbf{r}_j|},
\qquad
D(\theta) = \frac{2 J_1(k a \sin\theta)}{k a \sin\theta}$$

with $k = 2\pi/\lambda$, piston radius $a$, drive phase $\phi_j$.  The
total field is the coherent sum $p = \sum_j p_j$.

The particle velocity comes from the momentum equation
$\mathbf{v} = -\nabla p / (i\omega\rho_0)$, and the time-averaged
**Gor'kov potential** of a small sphere is

$$U = V_p\left[ f_1 \frac{\langle p^2\rangle}{2\rho_0 c_0^2} - f_2 \frac{3\rho_0}{4}\langle v^2\rangle \right],
\qquad
f_1 = 1 - \frac{\rho_0 c_0^2}{\rho_p c_p^2},\quad
f_2 = \frac{2(\rho_p - \rho_0)}{2\rho_p + \rho_0}$$

with radiation force $\mathbf{F} = -\nabla U$.  For solid/water particles
in air both $f_1, f_2 \approx 1$, so **traps sit at pressure nodes** where
$|p|$ and $|v|$ are simultaneously small.

**Limitations** — the model is linear (no acoustic streaming, no
multiple scattering, no viscous attenuation over the short path, no
finite-amplitude effects); transducers are idealised as pistons; the STL
mesh curvature/position of each socket is approximated by the analytical
spherical cap rather than read from the mesh.

### Phase-control cheat-sheet

| Effect | Recipe (per-transducer phase) |
|---|---|
| Standing wave (levitate) | $\phi_j = \mathrm{sign}(z_j)\cdot\pi/2 + \delta$ |
| Move up/down | change $\delta$ (top–bottom offset) |
| Focus (pressure max) | $\phi_j = -k\,|\mathbf{r}_t - \mathbf{r}_j|$ |
| Steer / translate | $\phi_j = -k\,(\mathbf{d}\cdot\mathbf{r}_j)$ |
| Rotate (vortex) | $\phi_j = m\,\mathrm{atan2}(y_j, x_j)$ |
